# 03 - Generators and Provider System

> **When to use**: When you need to choose a data generation engine (base/faker/mimesis), or understand the capabilities of 36 generators.
>
> **Core concept**: sqlseed supports 3 Providers, Mimesis (optional, preferred default), Faker (required by Core), and Base (built-in placeholders).

## Applicable Scenarios

- CI/CD using built-in placeholder values → use `base`
- Need rich localized data (Chinese names, addresses, etc.) → use `mimesis`
- Need specific format data (e.g., SSN, license plate) → use supported native provider methods
- Want to develop custom generators → implement `DataProvider` Protocol

## What You Will Learn

- 36 built-in generator types
- Capability differences and fallback strategy of 3 Providers
- locale localization support
- Custom Provider development

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| **→ 03** | **Generators and Provider System** | **Generators** | **01** |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---
## Setup

Use Python 3.10+ and run this notebook from `examples/notebooks` in a repository checkout. Select a notebook kernel from the environment containing these packages:

```bash
python -m pip install 'sqlseed[mimesis]==0.2.4' 'sqlseed-cli==0.2.4' jupyterlab
```

For source development, install Core and CLI together as described in the [repository README](../../README.md). This notebook creates its own temporary database and cache. Run cells from top to bottom; the validation helpers raise on partial generation or failed CLI commands.


In [ ]:
# Install the packages listed in Setup into the selected notebook kernel.
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
import os
import tempfile
from pathlib import Path
notebook_temp = tempfile.TemporaryDirectory(prefix="sqlseed-notebook-03-")
work_dir = Path(notebook_temp.name)
os.environ["SQLSEED_CACHE_DIR"] = str(work_dir / "cache")
db_path = build(work_dir / "demo.db")

# Fail visibly if a generation only partially succeeds.
generation_checks = []
def check_result(result, expected_count):
    if result.errors or result.count != expected_count:
        raise RuntimeError(f"{result.table_name}: expected {expected_count}, wrote {result.count}; errors={result.errors}")
    generation_checks.append({"table": result.table_name, "count": result.count, "errors": list(result.errors)})
    print(f"Verified {result.table_name}: {result.count} rows; errors={result.errors}")
    return result

def check_results(results, config_path):
    config = sqlseed.load_config(str(config_path))
    expected = {table.name: table.count for table in config.tables}
    for result in results:
        check_result(result, expected[result.table_name])
    if {result.table_name for result in results} != set(expected):
        raise RuntimeError("Not every configured table produced a result")
    return results

def check_cli(result):
    if result.exit_code != 0:
        raise RuntimeError(result.output) from result.exception
    return result


# Populate base dependencies
with connect(str(db_path)) as orch:
    check_result(orch.fill_table("organizations", count=5, seed=42), 5)
    check_result(orch.fill_table("members", count=20, seed=42), 20)
    check_result(orch.fill_table("projects", count=10, seed=42), 10)
    check_result(orch.fill_table("tags", count=8, seed=42), 8)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Generator Hub | `src/sqlseed/generators/registry.py` | `ProviderRegistry` |

> Corresponding architecture diagram: [§4 Data Generation Layer Architecture](../../docs/architecture.zh-CN.md#4-数据生成层架构)

## 1. See the Effect First — Three Providers Comparison

Same table, same column, the data quality difference between Providers is clear at a glance:

In [ ]:
for provider_name in ["base", "faker", "mimesis"]:
    rows = preview(str(db_path), table="members", count=2, provider=provider_name)
    sep = "=" * 70
    print()
    print(sep)
    print(f"  Provider: {provider_name}")
    print(sep)
    for row in rows:
        addr = str(row.get("address", "N/A"))[:40]
        print(f"  name={row.get('name', 'N/A'):<20s} email={row.get('email', 'N/A'):<30s}")
        print(f"  phone={row.get('phone', 'N/A'):<20s} address={addr}")

- **base**: built-in placeholder values; requires no additional provider package.
- **faker**: localized methods; Faker is a required Core dependency.
- **mimesis**: an optional provider with localized methods; install the Mimesis extra for these examples.

Compare generated values and measure performance on your schema rather than assuming a fixed speed ranking.

## 2. 36 Generators Overview

sqlseed has 36 built-in generators, listed below:

### Basic Types (9)

| Generator | Description | Key Params |
|--------|------|----------|
| `string` | random string | min_length, max_length, charset |
| `integer` | integer | min_value, max_value |
| `float` | float | min_value, max_value, precision |
| `boolean` | boolean | - |
| `bytes` | binary data | length |
| `json` | JSON data | - |
| `choice` | enum choice | choices |
| `weighted_choice` | weighted selection | choices or weighted_choices |
| `template` | formatted string | template, sequence_start, sequence_step |

### Semantic Types (17)

| Generator | Description | Example |
|--------|------|----------|
| `name` | name | 张三 / John Smith |
| `first_name` | first name | 伟 / John |
| `last_name` | last name | 王 / Smith |
| `username` | username | user_3847 |
| `email` | email | test@example.com |
| `phone` | phone | +1-555-0123 |
| `address` | address | 123 Main St |
| `city` | city | Beijing / New York |
| `country` | country | China / United States |
| `state` | state/province | California |
| `zip_code` | zip code | 10001 |
| `company` | company | Acme Inc |
| `job_title` | job title | Software Engineer |
| `url` | URL | https://example.com |
| `ipv4` | IP address | 192.168.1.1 |
| `uuid` | UUID | 550e8400-e29b-41d4... |
| `country_code` | country code | CN / US |

### Time/Text Types (10)

| Generator | Description | Example |
|--------|------|----------|
| `date` | date | 2024-01-15 |
| `datetime` | datetime | 2024-01-15 10:30:00 |
| `time` | time of day | 10:30:00 |
| `timestamp` | timestamp | 1705312200 |
| `text` | long text | Lorem ipsum... |
| `sentence` | sentence | The quick brown fox... |
| `word` | word | example |
| `catch_phrase` | business phrase | shared modular service |
| `password` | password | k8Xf2mPq |
| `pattern` | regex generation | PRJ-000123 |
`foreign_key` and `skip` are orchestration modes, outside the 36-name provider dispatch.


| Feature | BaseProvider | FakerProvider | MimesisProvider |
|---|---|---|---|
| Provider dependency | Built in | Required Faker package | Optional Mimesis package |
| Localization | Placeholder values | en_US, zh_CN, etc. | en, zh, etc. |
| Install | Included in Core | Included in Core | `python -m pip install "sqlseed[mimesis]==0.2.4"` |

All three providers implement the same dispatch interface; locale support and output formats differ.

## 3. 6 Representative Generators Deep Demo

### 3.1 email — Semantic Inference

The `email` generator generates different styles of email addresses based on locale.

In [ ]:
from sqlseed import preview

rows = preview(str(db_path), table="members", count=3, provider="mimesis")
for row in rows:
    print(f"email: {row['email']}")

### 3.2 pattern — Regex Generation

The `pattern` generator uses the `rstr` library to generate data from regex, suitable for fixed-format IDs.

In [ ]:
rows = preview(
    str(db_path),
    table="projects",
    count=5,
    columns={
        "project_no": {"type": "pattern", "regex": "PRJ-\\d{6}"},
    },
)
for row in rows:
    print(f"project_no: {row['project_no']}")

### 3.3 choice — Enum Selection

The `choice` generator randomly selects from given options, suitable for finite sets like status, type, etc.

In [ ]:
rows = preview(
    str(db_path),
    table="tasks",
    count=5,
    columns={
        "priority": {"type": "choice", "choices": [1, 2, 3, 4]},
        "status": {"type": "choice", "choices": [0, 1, 2, 3]},
    },
)
for row in rows:
    print(f"priority: {row['priority']}, status: {row['status']}")

### 3.4 null_ratio — Null Control

The `null_ratio` parameter controls the proportion of nulls generated (0.0-1.0).
The ratios are probabilities; a 20-row sample need not contain exactly the requested fraction of NULLs.


In [ ]:
import sqlite3

from sqlseed import ColumnConfig

# null_ratio needs to be passed via ColumnConfig object, using connect() API
with connect(str(db_path)) as orch:
    result = check_result(orch.fill_table("members", count=20, clear_before=True,
        column_configs=[
            ColumnConfig(name="phone", generator="phone", null_ratio=0.3),
            ColumnConfig(name="address", generator="address", null_ratio=0.5),
        ]), 20)


conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT phone, address FROM members").fetchall()
phone_null = sum(1 for r in rows if r[0] is None)
addr_null = sum(1 for r in rows if r[1] is None)
print(f"phone null: {phone_null}/20 ({phone_null/20*100:.0f}%)")
print(f"address null: {addr_null}/20 ({addr_null/20*100:.0f}%)")
conn.close()

### 3.5 faker.name — Provider Switching

Choose `provider="faker"` for this call. For supported native overrides, use `faker_method` with `native_params`; the method must match the active provider.

In [ ]:
rows = preview(
    str(db_path),
    table="members",
    count=3,
    provider="faker",
    locale="zh_CN",
)
for row in rows:
    print(f"name: {row['name']}, email: {row['email']}")
print("\nFaker + zh_CN locale generates Chinese names and emails")

### 3.6 mimesis.address — Mimesis Comparison

Mimesis is the preferred default provider when installed. Note the locale format differences:
- Faker: `en_US`, `zh_CN` (with underscore)
- Mimesis: `en`, `zh` (short code)

In [ ]:
rows = preview(
    str(db_path),
    table="members",
    count=3,
    provider="mimesis",
    locale="zh",
)
for row in rows:
    print(f"name: {row['name']}, address: {row.get('address', 'N/A')}")
print("\nMimesis + zh locale generates Chinese names and addresses")

## 4. Provider Availability

The orchestrator requests the configured provider. If it cannot be loaded, it logs a warning and falls back directly to Base; it does not try a Mimesis → Faker → Base chain. Install the desired optional provider explicitly instead of relying on fallback. Faker is already a required Core dependency.


In [ ]:
for provider_name in ["base", "faker", "mimesis"]:
    rows = preview(str(db_path), table="members", count=2, provider=provider_name)
    print(f"\n--- {provider_name} ---")
    for row in rows:
        print(f"  name={row['name']}, email={row['email']}")

## 5. locale and seed

### locale Settings

| Provider | Supported locale Format | Example |
|----------|-------------------|------|
| mimesis | short code | `en`, `zh`, `ja`, `de` |
| faker | underscore code | `en_US`, `zh_CN`, `ja_JP` |
| base | no localization | - |

### seed Reproducibility

In [ ]:
rows1 = preview(str(db_path), table="members", count=3, seed=42)
rows2 = preview(str(db_path), table="members", count=3, seed=42)

names1 = [r['name'] for r in rows1]
names2 = [r['name'] for r in rows2]

print(f"Run 1: {names1}")
print(f"Run 2: {names2}")
print(f"Reproducible: {names1 == names2}")
assert names1 == names2


## 🆕 bytes / json / timestamp Generators

These three generators are for special data types:

In [ ]:
with sqlseed.connect(str(db_path)) as orch:
    preview_bytes = orch.preview_table("organizations", count=3, columns={
        "description": {"generator": "bytes"},
    })
    print("bytes generator (raw bytes):")
    for row in preview_bytes:
        val = row.get('description')
        print(f"  description type: {type(val).__name__}, len: {len(val) if val else 0}")

preview_json = sqlseed.preview(str(db_path), table="organizations", count=2, columns={
    "description": {"generator": "json"}
})
print("\njson generator:")
for row in preview_json:
    print(f"  {row.get('description', 'N/A')}")

preview_ts = sqlseed.preview(str(db_path), table="organizations", count=2, columns={
    "created_at": {"generator": "timestamp"}
})
print("\ntimestamp generator:")
for row in preview_ts:
    print(f"  created_at: {row.get('created_at')}")

## 🔧 Underlying Provider Native Methods

sqlseed's Providers wrap the Faker and Mimesis libraries. Advanced users can access the underlying library's native methods via ProviderRegistry to get data types not covered by built-in generators (e.g., `license_plate`, `credit_card_number`, `food.fruit`, etc.).

In [ ]:
from sqlseed.generators.registry import ProviderRegistry

# Underlying Provider can call native methods directly
# This is advanced usage, generally columns={} config is sufficient
registry = ProviderRegistry()

# Faker native methods
registry.ensure_provider("faker")
faker_provider = registry.get("faker")
faker_provider.set_locale("en_US")
faker_obj = getattr(faker_provider, "_faker", None)

print("Faker native method examples:")
print(f"  company_suffix: {faker_obj.company_suffix()}")
print(f"  catch_phrase:   {faker_obj.catch_phrase()}")
print(f"  bs:             {faker_obj.bs()}")
print(f"  license_plate:  {faker_obj.license_plate()}")

# Mimesis native methods
registry.ensure_provider("mimesis")
mimesis_provider = registry.get("mimesis")
generic_obj = getattr(mimesis_provider, "_generic", None)

print("\nMimesis native method examples:")
print(f"  text.word:      {generic_obj.text.word()}")
print(f"  person.title:   {generic_obj.person.title()}")
print(f"  food.fruit:     {generic_obj.food.fruit()}")
print(f"  science.dna:    {generic_obj.science.dna_sequence()}")

## 📦 ProviderRegistry Complete API

ProviderRegistry manages all data generation Providers, supporting registration, query, setting defaults, etc.

In [ ]:
from sqlseed.generators.registry import ProviderRegistry

registry = ProviderRegistry()
print(f"Default Provider: {registry.default_name}")
print(f"Available Providers: {registry.available_providers}")

base = registry.get("base")
print(f"\nBaseProvider: name={base.name}")

registry.ensure_provider("faker")
print(f"After loading FakerProvider: {registry.available_providers}")

registry.ensure_provider("mimesis")
print(f"After loading MimesisProvider: {registry.available_providers}")

registry.set_default("faker")
print(f"Switch default to faker: default_name={registry.default_name}")

## 🔄 Provider Switching

Switch generation engine via the `provider` parameter of `preview()` / `fill()`. Different Providers have significant differences in data quality and localization support:

In [ ]:
# Provider switching via preview/fill's provider parameter
# Quality differences for same data type across different Providers:

print("Provider comparison — same column, different engines:\n")
print(f"{'Provider':<10s}  {'name':<25s}  {'email':<30s}")
print("-" * 68)

for prov in ["base", "faker", "mimesis"]:
    rows = preview(str(db_path), table="members", count=2, provider=prov)
    for row in rows:
        print(f"{prov:<10s}  {row['name']:<25s}  {row['email']:<30s}")
    print()

print("Choose a provider by the required locale and value formats; this notebook compares their actual output.")

## 6. Summary

| Key Point | Description |
|------|------|
| 36 generators | Basic 9 + Semantic 17 + Time/Text 10 |
| 3 Providers | Mimesis (optional preferred default), Faker, Base |
| Auto fallback | Warning and direct Base fallback when the requested provider is unavailable |
| locale | mimesis uses short codes, faker uses underscore codes |
| seed | Reproducible when set, suitable for testing |
| null_ratio | 0.0-1.0 controls null proportion |

**Next**: [04-database-advanced.ipynb](04-database-advanced.ipynb) — Database Layer and Multi-table

In [ ]:
# Verify exact database totals after the full notebook, plus every declared FK.
import sqlite3
expected_counts = {'organizations': 5, 'members': 20, 'projects': 10, 'tasks': 0, 'tags': 8, 'reviews': 0}
with sqlite3.connect(str(db_path)) as verification_db:
    actual_counts = {
        table: verification_db.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
        for table in expected_counts  # Fixed tutorial table names.
    }
    assert actual_counts == expected_counts, (actual_counts, expected_counts)
    fk_errors = verification_db.execute("PRAGMA foreign_key_check").fetchall()
    assert not fk_errors, fk_errors
print("Verified database row counts:", actual_counts)
print("Database FK check:", fk_errors)
print("Verified fill operations:", len(generation_checks))
